# Long Short-Term Memory (LSTM)
"Long Short-Term Memory" (Hochreiter & Schmidhuber, 1997), forget gate added by Gers et al. (2000)

**Idea:** add a separate **cell state** $c_t$ that is updated *additively* and controlled by learned **gates**. The additive path lets gradients flow over many steps without vanishing.

### LSTM vs RNN
| | RNN | LSTM |
|---|---|---|
| State | $h_t$ | $h_t$ (output) + $c_t$ (memory) |
| Update | $h_t = \tanh(W x_t + U h_{t-1})$, overwritten every step | $c_t = f_t \odot c_{t-1} + i_t \odot g_t$, **additive** |
| Gates | none | forget $f$, input $i$, output $o$ (sigmoids in $[0,1]$) |
| Gradient path back in time | product of $W_{hh}$ Jacobians, vanishes | through $c$: product of forget gates $f_t$, can stay $\approx 1$ |
| Params per layer | $H(E + H)$ | $4 \times$ that (4 gates/candidates) |
| Parallel over time? | no | no |

Shape flow: `idx (B, T) -> embed (B, T, E) -> loop t: (h, c) each (B, H) -> stack h (B, T, H) -> logits (B, T, V)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - tiny Shakespeare (character level)
~1.1M characters, vocab = 65 unique characters. Each training example is a random chunk of `block_size` characters; the target is the same chunk shifted by one (predict the next character).

The same data and loss are used in `rnn`, `lstm`, `gru`, `slm`, `moe`, `ssm`, so their val losses can be compared.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))          # (1115394,)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 64   # T: characters per sequence
batch_size = 64   # B

def get_batch(split):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print("vocab size:", vocab_size, "| train chars:", len(train_data), "| val chars:", len(val_data))

## 2. LSTM cell
All four pre-activations come from one `nn.Linear` on $[x_t, h_{t-1}]$, then are split.

### Formula: Gates
$$f_t = \sigma(W_f [x_t, h_{t-1}] + b_f), \quad i_t = \sigma(W_i [x_t, h_{t-1}] + b_i), \quad o_t = \sigma(W_o [x_t, h_{t-1}] + b_o)$$
- $[x_t, h_{t-1}] \in \mathbb{R}^{E+H}$: input and previous hidden state concatenated
- $\sigma$: sigmoid, outputs in $(0, 1)$, acts as a soft on/off switch per unit
- $f_t$: **forget** gate, how much of $c_{t-1}$ to keep
- $i_t$: **input** gate, how much of the new candidate to write
- $o_t$: **output** gate, how much of the memory to expose as $h_t$
- $W_f, W_i, W_o \in \mathbb{R}^{H \times (E+H)}$: fused with $W_g$ into `self.linear` ($4H$ outputs)

### Formula: Cell and hidden update
$$g_t = \tanh(W_g [x_t, h_{t-1}] + b_g), \qquad c_t = f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh(c_t)$$
- $g_t$: candidate values to write, in $(-1, 1)$
- $c_t \in \mathbb{R}^{H}$: cell state (long-term memory), no squashing between steps
- $\odot$: element-wise product
- $h_t \in \mathbb{R}^{H}$: hidden state / output (short-term, gated view of memory)

### Formula: Why it doesn't vanish
$$\frac{\partial c_t}{\partial c_{t-1}} = \text{diag}(f_t) \quad (\text{plus terms through } h_{t-1})$$
- if the model sets $f_t \approx 1$, the gradient passes back through $c$ almost unchanged: a "gradient highway"
- forget-gate bias is initialized to 1, so $f_t \approx \sigma(1) = 0.73$ at the start: remember by default

In [ ]:
class LSTMCell(nn.Module):
    # x_t (B, E), (h, c) each (B, H) -> (h_new, c_new) each (B, H)
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.linear = nn.Linear(input_size + hidden_size, 4 * hidden_size)  # W_i, W_f, W_g, W_o fused
        nn.init.ones_(self.linear.bias[hidden_size:2 * hidden_size])         # forget bias = 1

    def forward(self, x, state):
        h, c = state
        gates = self.linear(torch.cat([x, h], dim=1))  # (B, E+H) -> (B, 4H)
        i, f, g, o = gates.chunk(4, dim=1)             # each (B, H)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g                              # (B, H) additive memory update
        h = o * torch.tanh(c)                          # (B, H)
        return h, c

## 3. Character-level LSTM language model
Same as `CharRNN`, but the state is the pair $(h, c)$ and only $h$ goes to the output layer.

In [ ]:
class CharLSTM(nn.Module):
    # idx (B, T), state (h, c) or None -> logits (B, T, V), (h, c)
    def __init__(self, vocab_size, emb_dim=64, hidden_size=256):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.cell = LSTMCell(emb_dim, hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, idx, state=None):
        B, T = idx.shape
        if state is None:
            zeros = torch.zeros(B, self.hidden_size, device=idx.device)
            state = (zeros, zeros)
        x = self.embed(idx)                  # (B, T, E)
        outs = []
        for t in range(T):
            state = self.cell(x[:, t], state)
            outs.append(state[0])            # h_t (B, H)
        out = torch.stack(outs, dim=1)       # (B, T, H)
        return self.head(out), state         # (B, T, V), ((B, H), (B, H))

model = CharLSTM(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.3f}M  (~4x the RNN cell)")
logits, (h, c) = model(torch.zeros(2, 10, dtype=torch.long, device=device))
print(logits.shape, h.shape, c.shape)

## 4. Training
Identical loop to `rnn.ipynb`: same data, same steps, same learning rate, so the val losses can be compared directly.

### Formula: Next-token cross-entropy
$$\mathcal{L} = -\frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T} \log p_\theta(x_{t+1} \mid x_{\le t})$$
- $B, T$: batch size and sequence length
- $x_{t+1}$: the true next character
- $p_\theta(\cdot \mid x_{\le t})$: softmax over the vocab from the logits at position $t$
- a model that guesses uniformly scores $\log 65 \approx 4.17$; lower is better

### Formula: Gradient clipping (by global norm)
$$g \leftarrow g \cdot \min\!\left(1, \frac{c}{\lVert g \rVert_2}\right)$$
- $g$: all parameter gradients concatenated into one vector
- $\lVert g \rVert_2$: its L2 norm
- $c$: max norm (1.0); rescales the whole gradient, keeps its direction

In [ ]:
max_iters = 3000
eval_every = 500
lr = 3e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

def loss_fn(model, x, y):
    logits, _ = model(x)                                              # (B, T, V)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # (B*T, V) vs (B*T,) -> scalar

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s)).item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    x, y = get_batch("train")
    loss = loss_fn(model, x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 5. Generation
The state $(h, c)$ is carried forward exactly like $h$ in the RNN.

### Formula: Sampling with temperature
$$p_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}, \qquad x_{t+1} \sim \text{Categorical}(p)$$
- $z_i$: logit of character $i$
- $\tau$: temperature (0.8); $\tau < 1$ sharpens (safer), $\tau > 1$ flattens (more random), $\tau \to 0$ = greedy argmax

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)       # (1, len(start))
    logits, state = model(idx)                                # read the prompt into the state
    out = list(start)
    for _ in range(n):
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)  # (1, V)
        next_id = torch.multinomial(probs, 1)                  # (1, 1)
        out.append(itos[next_id.item()])
        logits, state = model(next_id, state)                  # feed only the new char + carried state
    return "".join(out)

print(generate(model))

## Notes
- Expect a lower val loss than the RNN (~1.5 here) and noticeably more word-like samples.
- LSTM was the default sequence model (translation, speech, handwriting) until Transformers (2017).
- Still sequential in time: training can't parallelize across the $T$ steps, which is what Transformers fixed.
- `nn.LSTM(emb_dim, hidden_size, batch_first=True)` is the same cell with a fused cuDNN kernel. PyTorch orders the gates as $i, f, g, o$, like here.
- **GRU** (next) merges the gates into 2 and drops the separate cell state.